# Занятие 3. REST API и JSON — демонстрационный блокнот

**Курс:** Автоматизированный сбор данных из интернета (код + no-code)

Блокнот рассчитан на последовательный прогон на лекции. Все примеры используют **публичные API без регистрации**, поэтому занятие не зависит от выдачи ключей.

### Содержание

| Блок | на что обратить внимание в блоке |
|---|---|
| 0. Подготовка окружения | сессия, таймауты, User-Agent |
| 1. Анатомия ответа |  статус, заголовки, JSON vs текст |
| 2. Коды состояния | как пайплайн реагирует на 200/404/429/5xx |
| 3. JSON → DataFrame |  `json_normalize`, `record_path`, `meta`, explode |
| 4. Пагинация | offset/limit и Link-заголовки |
| 5. Rate limit и повторы | `Retry`, `Retry-After`, backoff |
| 6. Аутентификация | ключ в заголовке, кэш OAuth2-токена (сухой прогон) |
| 7. GraphQL | один запрос вместо N |
| 8. No-code эквивалент |  тот же сценарий в n8n/Zapier |


## 0. Подготовка окружения

Единая сессия для всех примеров: keep-alive, честный User-Agent, обязательный таймаут. Отсутствие таймаута — самая частая ошибка студентов: скрипт зависает навсегда.

In [ ]:
import json, time, random, os
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

TIMEOUT = 15  # секунды: (connect, read) тоже допустимо — (5, 15)

def make_session(total_retries: int = 4) -> requests.Session:
    """Сессия с повторами для 429 и 5xx и уважением к Retry-After."""
    s = requests.Session()
    retry = Retry(
        total=total_retries,
        backoff_factor=1.0,                 # 0с, 2с, 4с, 8с ...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "HEAD"],
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry, pool_maxsize=10))
    s.headers.update({
        "User-Agent": "course-data-collector/1.0 (educational use)",
        "Accept": "application/json",
    })
    return s

session = make_session()
print("Сессия готова:", session.headers["User-Agent"])

Сессия готова: course-data-collector/1.0 (educational use)


## 1. Анатомия HTTP-ответа

Демонстрация содержимого респонса на трех уровнях:
* строка состояния → заголовки → тело.
* ⚠️**Обращаем внимание** на `Content-Type`: именно он говорит, можно ли вызывать `.json()`.

In [ ]:
r = session.get("https://dummyjson.com/products", params={"limit": 3, "skip": 0}, timeout=TIMEOUT)

print("URL       :", r.url)
print("Статус    :", r.status_code, r.reason)
print("Тип       :", r.headers.get("Content-Type"))
print("Размер    :", len(r.content), "байт")
print("Время     :", round(r.elapsed.total_seconds(), 3), "с")
print("\nИнтересные заголовки:")
for h in ("Server", "Date", "Cache-Control", "ETag", "X-RateLimit-Remaining"):
    if h in r.headers:
        print(f"  {h}: {r.headers[h]}")

payload = r.json()
print("\nКлючи верхнего уровня:", list(payload))
print("Всего записей на сервере:", payload["total"])
print("\nПервый товар (усечённо):")
print(json.dumps(payload["products"][0], ensure_ascii=False, indent=2)[:700], "...")

URL       : https://dummyjson.com/products?limit=3&skip=0
Статус    : 200 OK
Тип       : application/json; charset=utf-8
Размер    : 4543 байт
Время     : 0.087 с

Интересные заголовки:
  Server: cloudflare
  Date: Tue, 22 Sep 2026 09:21:45 GMT
  ETag: W/"11bf-tlldaNxTnp0hDgZqUroOj8H12o8"
  X-RateLimit-Remaining: 99

Ключи верхнего уровня: ['products', 'total', 'skip', 'limit']
Всего записей на сервере: 194

Первый товар (усечённо):
{
  "id": 1,
  "title": "Essence Mascara Lash Princess",
  "description": "The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.",
  "category": "beauty",
  "price": 9.99,
  "discountPercentage": 10.48,
  "rating": 2.56,
  "stock": 99,
  "tags": [
    "beauty",
    "mascara"
  ],
  "brand": "Essence",
  "sku": "BEA-ESS-ESS-001",
  "weight": 4,
  "dimensions": {
    "width": 15.14,
    "height": 13.08,
    "depth": 22.99
  },
  "warrant

### Защитный разбор ответа

Правило пайплайна: никогда не вызываем `.json()` без проверки статуса и типа содержимого. При ошибке разбора сохраняем первые байты тела — это единственный способ потом понять, что вернул сервер (часто это HTML-страница блокировки).

In [ ]:
def get_json(session, url, **kwargs):
    """Возвращает (data, response). Бросает информативное исключение."""
    resp = session.get(url, timeout=TIMEOUT, **kwargs)
    if resp.status_code >= 400:
        raise requests.HTTPError(
            f"{resp.status_code} {resp.reason} for {resp.url}; тело: {resp.text[:200]!r}"
        )
    ctype = resp.headers.get("Content-Type", "")
    if "json" not in ctype:
        raise ValueError(f"Ожидался JSON, получен {ctype!r}; тело: {resp.text[:200]!r}")
    return resp.json(), resp

data, _ = get_json(session, "https://dummyjson.com/products/1")
print(data["title"], "|", data["price"], "|", data["category"])

# демонстрация ошибки: несуществующий ресурс
try:
    get_json(session, "https://dummyjson.com/products/999999")
except requests.HTTPError as e:
    print("\nОжидаемая ошибка:", str(e)[:160])

Essence Mascara Lash Princess | 9.99 | beauty

Ожидаемая ошибка: 404 Not Found for https://dummyjson.com/products/999999; тело: '{"message":"Product with id \'999999\' not found"}'


## 2. Коды состояния как таблица решений

Показываем на живых запросах и сразу связываем код с действием пайплайна.
* ⚠️Главная мысль: 4xx (кроме 429) повторять бессмысленно, 429 и 5xx — повторять с задержкой.

In [ ]:
PROBES = [
    ("https://httpbin.org/status/200", "успех"),
    ("https://httpbin.org/status/404", "нет ресурса"),
    ("https://httpbin.org/status/429", "лимит"),
    ("https://httpbin.org/status/503", "сбой сервера"),
]

def decision(code: int) -> str:
    if 200 <= code < 300:  return "обработать данные"
    if code in (301, 302, 307, 308): return "следовать редиректу, сохранить новый URL"
    if code == 304: return "взять из локального кэша"
    if code == 429: return "ждать Retry-After, затем повтор с backoff"
    if 500 <= code:  return "повтор с backoff, максимум N попыток"
    if code in (401, 403): return "проверить токен/права, НЕ повторять слепо"
    if 400 <= code:  return "исправить запрос, не повторять"
    return "разобрать вручную"

plain = make_session(total_retries=0)   # без автоповторов, чтобы увидеть коды как есть
rows = []
for url, note in PROBES:
    try:
        resp = plain.get(url, timeout=TIMEOUT)
        rows.append({"код": resp.status_code, "смысл": note, "действие": decision(resp.status_code)})
    except requests.RequestException as e:
        rows.append({"код": "—", "смысл": note, "действие": f"сетевая ошибка: {type(e).__name__}"})

display(pd.DataFrame(rows))

,код,смысл,действие
0,200,успех,обработать данные
1,404,нет ресурса,"исправить запрос, не повторять"
2,429,лимит,"ждать Retry-After, затем повтор с backoff"
3,503,сбой сервера,"повтор с backoff, максимум N попыток"


## 3. JSON → DataFrame

Три типовые формы ответа, которые встречаются в 90% задач курса.

**3.1. Список объектов с вложенностью** — `json_normalize` с `sep = "_"` автоматически разворачивает все вложенные словари в плоские колонки. Например, поле  `dimensions.width` разворачивает в колонку `dimensions_width`.

In [ ]:
data, _ = get_json(session, "https://dummyjson.com/products", params={"limit": 30})
products = data["products"]

flat = pd.json_normalize(products, sep="_")
print("Форма:", flat.shape)
print("Колонки:", list(flat.columns)[:18], "...")
display(flat[["id", "title", "category", "price", "rating",
              "dimensions_width", "dimensions_height", "meta_createdAt"]].head())

Форма: (30, 27)
Колонки: ['id', 'title', 'description', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'reviews', 'returnPolicy', 'minimumOrderQuantity'] ...


,id,title,category,price,rating,dimensions_width,dimensions_height,meta_createdAt
0,1,Essence Mascara Lash Princess,beauty,9.99,2.56,15.14,13.08,2025-10-09T14:47:01.588Z
1,2,Eyeshadow Palette with Mirror,beauty,19.99,2.86,9.26,22.47,2024-12-03T11:29:45.706Z
2,3,Powder Canister,beauty,14.99,4.64,29.27,27.93,2025-01-08T18:30:12.844Z
3,4,Red Lipstick,beauty,12.99,4.36,18.11,28.38,2025-11-22T23:28:00.911Z
4,5,Red Nail Polish,beauty,8.99,4.32,21.63,16.48,2025-10-03T07:39:53.470Z


**3.2. Вложенный массив → отдельные строки.** `record_path` задаёт массив, который становится строками, `meta` — поля родителя, которые нужно протянуть. Это ровно та операция, которую студенты обычно пишут тройным циклом.

In [ ]:
reviews = pd.json_normalize(
    products,
    record_path="reviews",
    meta=["id", "title", "category"],
    meta_prefix="product_",
    record_prefix="review_",
    errors="ignore",          # у части товаров ключа может не быть
)
print("Отзывов:", len(reviews))
display(reviews.head(6))

print("\nСредняя оценка по категориям:")
display(reviews.groupby("product_category")["review_rating"].agg(["count", "mean"]).round(2))

Отзывов: 90


,review_rating,review_comment,review_date,review_reviewerName,review_reviewerEmail,product_id,product_title,product_category
0,3,Would not recommend!,2025-04-30T09:41:02.053Z,Eleanor Collins,eleanor.collins@x.dummyjson.com,1,Essence Mascara Lash Princess,beauty
1,4,Very satisfied!,2025-04-30T09:41:02.053Z,Lucas Gordon,lucas.gordon@x.dummyjson.com,1,Essence Mascara Lash Princess,beauty
2,5,Highly impressed!,2025-04-30T09:41:02.053Z,Eleanor Collins,eleanor.collins@x.dummyjson.com,1,Essence Mascara Lash Princess,beauty
3,5,Great product!,2025-04-30T09:41:02.053Z,Savannah Gomez,savannah.gomez@x.dummyjson.com,2,Eyeshadow Palette with Mirror,beauty
4,4,Awesome product!,2025-04-30T09:41:02.053Z,Christian Perez,christian.perez@x.dummyjson.com,2,Eyeshadow Palette with Mirror,beauty
5,1,Poor quality!,2025-04-30T09:41:02.053Z,Nicholas Bailey,nicholas.bailey@x.dummyjson.com,2,Eyeshadow Palette with Mirror,beauty



Средняя оценка по категориям:


,count,mean
product_category,,
beauty,15,3.60
fragrances,15,3.93
furniture,15,3.27
groceries,45,3.62


**3.3. Параллельные массивы** (Open-Meteo, многие финансовые API): время и значения лежат в разных списках одинаковой длины. `json_normalize` здесь не поможет — собираем DataFrame напрямую и проверяем совпадение длин.

In [ ]:
wx, _ = get_json(session, "https://api.open-meteo.com/v1/forecast", params={
    "latitude": 56.85, "longitude": 60.60,          # Екатеринбург
    "hourly": "temperature_2m,relative_humidity_2m",
    "forecast_days": 2, "timezone": "Asia/Yekaterinburg",
})
h = wx["hourly"]
assert len({len(v) for v in h.values()}) == 1, "Длины массивов разошлись — данные испорчены"

wdf = pd.DataFrame(h)
wdf["time"] = pd.to_datetime(wdf["time"])
print("Единицы измерения:", wx["hourly_units"])
display(wdf.head())
print("\nСуточный минимум/максимум:")
display(wdf.set_index("time")["temperature_2m"].resample("1D").agg(["min", "max"]))

Единицы измерения: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%'}


,time,temperature_2m,relative_humidity_2m
0,2026-09-22 00:00:00,11.8,87
1,2026-09-22 01:00:00,11.3,89
2,2026-09-22 02:00:00,10.9,90
3,2026-09-22 03:00:00,10.6,92
4,2026-09-22 04:00:00,10.2,93



Суточный минимум/максимум:


,min,max
time,,
2026-09-22,9.8,21.6
2026-09-23,9.3,20.7


### Разреженные поля и качество данных

Open Library — удобный пример «грязного» API: у части записей нет автора, года или количества страниц. Сразу приучаем считать полноту полей, а не надеяться на неё.

In [ ]:
ol, _ = get_json(session, "https://openlibrary.org/search.json", params={
    "q": "machine learning", "limit": 50,
    "fields": "key,title,author_name,first_publish_year,language,number_of_pages_median",
})
books = pd.json_normalize(ol["docs"])
books["author_name"] = books.get("author_name").apply(
    lambda v: ", ".join(v) if isinstance(v, list) else None)

print("Найдено всего:", ol["numFound"], "| загружено:", len(books))
completeness = (books.notna().mean() * 100).round(1).sort_values(ascending=False)
print("\nПолнота полей, %:")
print(completeness)
display(books.head())

Найдено всего: 7356 | загружено: 50

Полнота полей, %:
author_name               100.0
first_publish_year        100.0
key                       100.0
title                     100.0
language                   94.0
number_of_pages_median     88.0
dtype: float64


,author_name,first_publish_year,key,language,number_of_pages_median,title
0,Aurélien Géron,2019,/works/OL20709638W,[eng],856.0,"Hands-On Machine Learning with Scikit-Learn, K..."
1,"Kevin P. Murphy, Kevin P. Murphy",2012,/works/OL16571885W,[eng],1104.0,Machine learning
2,Ethem Alpaydin,2016,/works/OL19723604W,[eng],224.0,Machine learning
3,Anil Ananthaswamy,2024,/works/OL37831042W,[eng],480.0,Why Machines Learn
4,"Andreas C. Mueller, Sarah Guido",2016,/works/OL17357597W,[eng],376.0,Introduction to Machine Learning with Python


## 4. Пагинация

**4.1. Offset / limit.** Обязательные предохранители: лимит страниц, проверка пустой страницы, пауза между запросами. Обратите внимание, что сервер может урезать запрошенный `limit` — цикл должен опираться на фактическую длину страницы.

In [ ]:
def fetch_paged_offset(session, url, page_size=30, max_records=150, pause=0.3,
                       items_key="products", total_key="total"):
    """Сбор с offset/limit-пагинацией. Возвращает список записей и журнал запросов."""
    out, log, skip, page = [], [], 0, 0
    while len(out) < max_records and page < 50:          # предохранитель
        data, resp = get_json(session, url, params={"limit": page_size, "skip": skip})
        items = data.get(items_key, [])
        log.append({"страница": page, "skip": skip, "получено": len(items),
                    "код": resp.status_code, "сек": round(resp.elapsed.total_seconds(), 2)})
        if not items:
            break
        out.extend(items)
        skip += len(items)                                # НЕ page_size: сервер мог урезать
        page += 1
        if total_key in data and skip >= data[total_key]:
            break
        time.sleep(pause)                                 # вежливость к серверу
    return out[:max_records], pd.DataFrame(log)

items, log = fetch_paged_offset(session, "https://dummyjson.com/products", page_size=30, max_records=120)
print("Собрано записей:", len(items), "| уникальных id:", len({i["id"] for i in items}))
display(log)

Собрано записей: 120 | уникальных id: 120


,страница,skip,получено,код,сек
0,0,0,30,200,0.01
1,1,30,30,200,0.01
2,2,60,30,200,0.01
3,3,90,30,200,0.01


**4.2. Cursor / Link-заголовки.** GitHub отдаёт ссылку на следующую страницу в заголовке `Link`; `requests` разбирает его в `resp.links`. Цикл идёт «пока есть rel=next» и не знает общего числа страниц заранее.

In [ ]:
def fetch_paged_link(session, url, params=None, max_pages=3, pause=0.5):
    rows, page = [], 0
    while url and page < max_pages:
        resp = session.get(url, params=params, timeout=TIMEOUT)
        if resp.status_code == 403 and resp.headers.get("X-RateLimit-Remaining") == "0":
            print("Квота GitHub исчерпана — пропускаем демонстрацию")
            break
        resp.raise_for_status()
        batch = resp.json()
        rows.extend(batch)
        print(f"страница {page}: +{len(batch)} записей | "
              f"остаток квоты: {resp.headers.get('X-RateLimit-Remaining')}")
        url = resp.links.get("next", {}).get("url")   # <-- разобранный заголовок Link
        params = None                                  # параметры уже внутри next-URL
        page += 1
        time.sleep(pause)
    return rows

issues = fetch_paged_link(
    session, "https://api.github.com/repos/scrapy/scrapy/issues",
    params={"state": "closed", "per_page": 30},
)
if issues:
    idf = pd.json_normalize(issues, sep="_")[
        ["number", "title", "state", "created_at", "closed_at", "user_login", "comments"]]
    idf["created_at"] = pd.to_datetime(idf["created_at"])
    print("Всего собрано:", len(idf))
    display(idf.head())

страница 0: +30 записей | остаток квоты: 47


страница 1: +30 записей | остаток квоты: 46


страница 2: +30 записей | остаток квоты: 45


Всего собрано: 90


,number,title,state,created_at,closed_at,user_login,comments
0,8209,docs: recipe for deduplicating query params in...,closed,2026-09-22 06:42:14+00:00,2026-09-22T07:50:06Z,AdrianAtZyte,1
1,8208,Add a two-legged OAuth request-signing recipe,closed,2026-09-22 05:20:18+00:00,2026-09-22T06:18:35Z,AdrianAtZyte,1
2,8195,Avoid decoding response bodies for html/xml se...,closed,2026-09-21 16:00:23+00:00,2026-09-21T17:56:25Z,AdrianAtZyte,2
3,8192,Never fail decoding headers in to_unicode_dict...,closed,2026-09-21 14:34:59+00:00,2026-09-21T15:15:32Z,AdrianAtZyte,2
4,8188,Claude now supports AGENTS.md,closed,2026-09-21 08:09:13+00:00,2026-09-21T08:16:41Z,AdrianAtZyte,0


## 5. Rate limit, повторы и инкрементальность

**5.1. Токен-бакет** — простой ограничитель скорости, который студенты смогут переиспользовать в лабораторной работе.

In [ ]:
class RateLimiter:
    """Не более `rate` запросов в секунду (равномерно)."""
    def __init__(self, rate_per_sec: float):
        self.interval = 1.0 / rate_per_sec
        self._last = 0.0
    def wait(self):
        delta = time.monotonic() - self._last
        if delta < self.interval:
            time.sleep(self.interval - delta)
        self._last = time.monotonic()

limiter = RateLimiter(rate_per_sec=2)
t0 = time.monotonic()
for i in range(4):
    limiter.wait()
    r = session.get("https://dummyjson.com/products/1", timeout=TIMEOUT)
    print(f"запрос {i+1}: {r.status_code}, t = {time.monotonic() - t0:.2f} c")

запрос 1: 200, t = 0.01 c


запрос 2: 200, t = 0.51 c


запрос 3: 200, t = 1.01 c


запрос 4: 200, t = 1.51 c


**5.2. Экспоненциальная задержка с джиттером.**

 Демонстрация, почему нужен джиттер: без него все клиенты повторяют синхронно и снова роняют сервис.

In [ ]:
def backoff_delays(attempts=6, base=1.0, cap=60.0, jitter=True):
    out = []
    for n in range(attempts):
        d = min(cap, base * 2 ** n)
        out.append(round(random.uniform(0, d) if jitter else d, 2))
    return out

print("Без джиттера:", backoff_delays(jitter=False))
print("С джиттером :", backoff_delays(jitter=True))

def request_with_retry(session, url, attempts=4, **kw):
    """Ручная реализация — полезна, когда нужно логировать каждую попытку."""
    for n in range(attempts):
        try:
            resp = session.get(url, timeout=TIMEOUT, **kw)
        except requests.RequestException as e:
            print(f"  попытка {n+1}: сетевая ошибка {type(e).__name__}")
        else:
            if resp.status_code < 400:
                return resp
            if resp.status_code in (429, 500, 502, 503, 504):
                wait = float(resp.headers.get("Retry-After", min(30, 2 ** n)))
                print(f"  попытка {n+1}: {resp.status_code}, ждём {wait:.1f} c")
                time.sleep(min(wait, 3))     # на лекции урезаем ожидание
            else:
                raise requests.HTTPError(f"{resp.status_code} — повтор не имеет смысла")
    raise RuntimeError(f"не удалось получить {url} за {attempts} попыток")

plain0 = make_session(total_retries=0)
try:
    request_with_retry(plain0, "https://httpbin.org/status/503", attempts=3)
except RuntimeError as e:
    print("Итог:", e)

Без джиттера: [1.0, 2.0, 4.0, 8.0, 16.0, 32.0]
С джиттером : [0.69, 0.34, 2.06, 4.94, 1.52, 2.66]
  попытка 1: 503, ждём 1.0 c


  попытка 2: 503, ждём 2.0 c


  попытка 3: 503, ждём 4.0 c


Итог: не удалось получить https://httpbin.org/status/503 за 3 попыток


**5.3. Инкрементальный сбор.**

Храним «водяной знак» — максимальную дату обновления из прошлого запуска — и запрашиваем только новое. Экономит квоту и делает ежедневный запуск дешёвым.

In [ ]:
STATE_FILE = "state_lecture3.json"

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {"watermark": None, "runs": 0}

def save_state(state):
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()
print("Состояние до запуска:", state)

fresh = [p for p in products
         if state["watermark"] is None or p["meta"]["updatedAt"] > state["watermark"]]
print("Новых/изменённых записей:", len(fresh))

if fresh:
    state["watermark"] = max(p["meta"]["updatedAt"] for p in fresh)
state["runs"] += 1
save_state(state)
print("Состояние после запуска:", state)
print("Повторный прогон этой ячейки даст 0 новых записей — это и есть идемпотентность.")

Состояние до запуска: {'watermark': None, 'runs': 0}
Новых/изменённых записей: 30
Состояние после запуска: {'watermark': '2026-08-22T08:09:35.091Z', 'runs': 1}
Повторный прогон этой ячейки даст 0 новых записей — это и есть идемпотентность.


## 6. Аутентификация

Живой ключ на лекции не нужен: показываем, **как** передаётся секрет и почему он не должен попадать в URL и репозиторий. `httpbin.org` возвращает полученные заголовки, что удобно для демонстрации.

In [ ]:
API_KEY = os.getenv("DEMO_API_KEY", "demo-key-not-a-real-secret")

r = session.get("https://httpbin.org/headers",
                headers={"Authorization": f"Bearer {API_KEY}", "X-API-Key": API_KEY},
                timeout=TIMEOUT)
received = r.json()["headers"]
print("Сервер увидел:")
for k in ("Authorization", "X-Api-Key", "User-Agent"):
    if k in received:
        print(f"  {k}: {received[k][:24]}…")

print("""
Правила курса:
  1) секрет — в переменной окружения или .env (в .gitignore), не в коде;
  2) заголовок, а не query-параметр: URL попадает в логи прокси и историю;
  3) в отчёте по лабораторной работе ключ маскируется;
  4) один ключ на студента — не шарить между командами.""")

Сервер увидел:
  Authorization: Bearer demo-key-not-a-re…
  X-Api-Key: demo-key-not-a-real-secr…
  User-Agent: course-data-collector/1.…

Правила курса:
  1) секрет — в переменной окружения или .env (в .gitignore), не в коде;
  2) заголовок, а не query-параметр: URL попадает в логи прокси и историю;
  3) в отчёте по лабораторной работе ключ маскируется;
  4) один ключ на студента — не шарить между командами.


**OAuth2 client credentials — сухой прогон.**

Логика кэширования токена важнее конкретного провайдера: обновляем заранее, а не по 401.

In [ ]:
class TokenCache:
    def __init__(self, token_url, client_id, client_secret, skew=60):
        self.token_url, self.cid, self.secret, self.skew = token_url, client_id, client_secret, skew
        self._token, self._expires_at = None, 0.0
        self.requests_made = 0
    def get(self):
        if self._token and time.time() < self._expires_at - self.skew:
            return self._token          # попадание в кэш — сетевого запроса нет
        self._token, ttl = self._request_token()
        self._expires_at = time.time() + ttl
        return self._token
    def _request_token(self):
        # Реальный вызов:
        # resp = requests.post(self.token_url, data={
        #     "grant_type": "client_credentials",
        #     "client_id": self.cid, "client_secret": self.secret}, timeout=15)
        # resp.raise_for_status(); p = resp.json()
        # return p["access_token"], p.get("expires_in", 3600)
        self.requests_made += 1
        return f"fake-token-{self.requests_made}", 3600

tc = TokenCache("https://example.com/oauth/token", "client-id", "client-secret")
print([tc.get() for _ in range(3)])
print("Запросов к серверу токенов:", tc.requests_made, "— остальные из кэша")

['fake-token-1', 'fake-token-1', 'fake-token-1']
Запросов к серверу токенов: 1 — остальные из кэша


## 7. GraphQL: один запрос вместо N

REST-вариант «репозиторий + его последние issue» требует минимум двух запросов и приносит десятки лишних полей. GraphQL-запрос описывает ровно нужные поля. Показываем текст запроса и считаем экономию; живой вызов требует токена, поэтому он закомментирован.

In [ ]:
GQL = """
query RepoSnapshot($owner: String!, $name: String!, $n: Int!) {
  repository(owner: $owner, name: $name) {
    stargazerCount
    forkCount
    issues(first: $n, states: OPEN, orderBy: {field: CREATED_AT, direction: DESC}) {
      totalCount
      pageInfo { hasNextPage endCursor }
      nodes { number title createdAt author { login } }
    }
  }
}
"""
variables = {"owner": "scrapy", "name": "scrapy", "n": 5}
print(GQL)
print("Переменные:", variables)

# Живой вызов (нужен токен с правом read:public_repo):
# resp = requests.post("https://api.github.com/graphql",
#                      json={"query": GQL, "variables": variables},
#                      headers={"Authorization": f"Bearer {os.environ['GH_TOKEN']}"}, timeout=15)
# body = resp.json()
# if "errors" in body: raise RuntimeError(body["errors"])   # код 200 при ошибке — норма для GraphQL
# data = body["data"]["repository"]

rest = session.get("https://api.github.com/repos/scrapy/scrapy", timeout=TIMEOUT)
if rest.status_code == 200:
    print(f"\nREST-ответ по одному репозиторию: {len(rest.json())} полей, {len(rest.content)} байт —")
    print("из них в задаче нужны 2. GraphQL вернул бы только запрошенные поля за 1 запрос.")


query RepoSnapshot($owner: String!, $name: String!, $n: Int!) {
  repository(owner: $owner, name: $name) {
    stargazerCount
    forkCount
    issues(first: $n, states: OPEN, orderBy: {field: CREATED_AT, direction: DESC}) {
      totalCount
      pageInfo { hasNextPage endCursor }
      nodes { number title createdAt author { login } }
    }
  }
}

Переменные: {'owner': 'scrapy', 'name': 'scrapy', 'n': 5}

REST-ответ по одному репозиторию: 86 полей, 6168 байт —
из них в задаче нужны 2. GraphQL вернул бы только запрошенные поля за 1 запрос.


## 8. No-code эквивалент того же сценария

Итог занятия: один и тот же пайплайн «API → нормализация → таблица», собранный тремя способами.

**Python (то, что мы только что сделали):** `make_session` → `fetch_paged_offset` → `json_normalize` → CSV/БД. Полный контроль, версионирование в Git, тесты; нужна среда исполнения и расписание.

**n8n (low-code, self-hosted):**
1. `Schedule Trigger` — каждый день в 06:00.
2. `HTTP Request` — URL, query-параметры, предустановка аутентификации, встроенная пагинация (Response → Pagination: `skip` += `limit`, стоп по пустому ответу).
3. `Split Out` — развернуть массив `products` в отдельные items.
4. `Set` / `Code` — переименовать и типизировать поля (аналог `json_normalize`).
5. `Loop Over Items` + `Wait` — батчи и контроль скорости (аналог `RateLimiter`).
6. `Postgres` / `Google Sheets` — выгрузка; ветка ошибки узла → уведомление.

**Zapier (no-code):** `Schedule` → `Webhooks by Zapier (GET)` → `Formatter (Utilities / Line-item to text)` → `Google Sheets: Create Row`. Собирается за 15–20 минут, но каждый шаг тарифицируется как задача, а полноценного цикла по страницам нет — пагинация делается ветвлениями или вовсе недоступна.

## Итоги и переход к лабораторной работе

Проверьте, что каждый студент унёс с занятия:

1. Шаблон `make_session` с таймаутом, User-Agent и повторами.
2. Функцию `get_json` с проверкой статуса и `Content-Type`.
3. Два шаблона пагинации: offset/limit и Link/cursor — оба с предохранителем.
4. Понимание `json_normalize(record_path=..., meta=...)`.
5. Привычку не хранить секреты в коде.
6. Умение обосновать выбор «код vs no-code» по критериям, а не по привычке.

**Домашнее/лабораторное задание (6 часов):** собрать 300+ записей из двух источников (API с пагинацией + второй API для обогащения), нормализовать, проверить качество, повторить один сценарий в no-code и оформить отчёт. Полное условие — в файле задания на лабораторную работу.

### Типичные ошибки студентов (предупредить заранее)

| Ошибка | Последствие | Что требовать |
|---|---|---|
| Нет `timeout` | скрипт висит вечно | таймаут в каждом запросе |
| Повтор при 400/401 | бан по ключу | повторять только 429 и 5xx |
| `skip += page_size` вместо фактической длины | пропуски записей | инкремент по `len(items)` |
| Нет предела страниц | бесконечный цикл | явный `max_pages` |
| Ключ в коде/репозитории | утечка секрета | переменные окружения |
| `d["key"]` на разреженных данных | падение на 200-й записи | `.get()` и подсчёт полноты |
| Сбор без паузы | 429 и блокировка | `RateLimiter` или `time.sleep` |